# Contrastive Predictive Coding (CPC v2)

_Semi & Self-Supervised Learning_

**Learn representations by predicting the future in latent space — and judge the prediction by picking the true future out of a crowd of negatives.**

Suppose nobody labels your data. How do you still learn a useful representation? CPC (Contrastive Predictive Coding) has a clever answer: predict the future.

       Not the raw future pixels — those are noisy and full of detail you do not care about. Instead, predict the future in latent space: a short summary vector of what comes next.

---

This notebook is a practice scaffold for the **Contrastive Predictive Coding (CPC v2)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CPC(nn.Module):
    def __init__(self, in_dim=1, latent=64, ctx=64, n_pred=4):
        super().__init__()
        self.n_pred = n_pred                          # predict k = 1..n_pred steps ahead
        # encoder g_enc: each timestep x_t -> latent z_t
        self.enc = nn.Sequential(
            nn.Linear(in_dim, 128), nn.LayerNorm(128), nn.ReLU(),
            nn.Linear(128, latent),
        )
        # autoregressive context g_ar: summarize z_{<=t} into c_t
        self.ar = nn.GRU(latent, ctx, batch_first=True)
        # one prediction head W_k per future offset k
        self.Wk = nn.ModuleList([nn.Linear(ctx, latent, bias=False)
                                 for _ in range(n_pred)])

    def forward(self, x):
        # x: (B, T, in_dim) — B sequences of length T
        B, T, _ = x.shape
        z = self.enc(x)                               # (B, T, latent)
        c, _ = self.ar(z)                             # (B, T, ctx) contexts at every t

        loss = 0.0
        for k in range(1, self.n_pred + 1):
            t_max = T - k                             # last anchor that has a real future
            ct = c[:, :t_max, :]                      # (B, t_max, ctx) contexts
            z_true = z[:, k:k + t_max, :]             # (B, t_max, latent) true z_{t+k}
            pred = self.Wk[k - 1](ct)                 # (B, t_max, latent) = W_k c_t

            # flatten (B * t_max) into one pool; every other latent is a negative
            P = pred.reshape(-1, pred.size(-1))       # (M, latent) predictions
            Z = z_true.reshape(-1, z_true.size(-1))   # (M, latent) true futures = candidates
            P = F.normalize(P, dim=1)
            Z = F.normalize(Z, dim=1)

            logits = P @ Z.t()                        # (M, M) score every pred vs every candidate
            # row i's correct candidate is its own true future, on the diagonal
            targets = torch.arange(P.size(0), device=x.device)
            loss = loss + F.cross_entropy(logits / 0.1, targets)   # InfoNCE for offset k

        return loss / self.n_pred

# toy run: 8 sequences, length 20, scalar signal per step
model = CPC(in_dim=1, n_pred=4)
x = torch.randn(8, 20, 1)
loss = model(x)
loss.backward()                                       # gradients flow into enc, GRU, and W_k
print(float(loss))

## Visualize the data & results

_On real digit images read as a sequence of 8 pixel-rows, can a context vector predict the TRUE next row's latent out of N negatives by cosine InfoNCE — and how does that get harder as we predict further ahead or face more negatives?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

# Read each 8x8 digit as a SEQUENCE of 8 pixel-ROWS. Encode each row to a latent (PCA),
# learn a per-gap linear prediction head W_k (CPC's z_{t+k} ~ W_k c_t), then measure top-1
# contrastive accuracy: pick the TRUE row-(t+k) latent out of N negatives by cosine
# similarity to W_k c_t. Average over every valid anchor t to isolate the GAP effect.

rng = np.random.default_rng(0)
X = load_digits().data.reshape(-1, 8, 8) / 16.0   # (1797, 8 rows, 8 cols), real images
N_IMG = X.shape[0]

D = 8
rows = X.reshape(-1, 8)                            # one row = one timestep
Z = PCA(n_components=D, random_state=0).fit_transform(rows).reshape(N_IMG, 8, D)

perm = rng.permutation(N_IMG)
tr, te = perm[:1200], perm[1200:]                 # split images train / test

def unit(a):
    return a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-9)

def fit_head(t, k):                               # ridge-regress z_{t+k} from z_t
    Ctr, Ztr = Z[tr, t, :], Z[tr, t + k, :]
    A = Ctr.T @ Ctr + 1e-2 * np.eye(D)
    return np.linalg.solve(A, Ctr.T @ Ztr)        # (D, D) prediction head W_k

def accuracy(k, n_neg, n_trials_per=1500):
    hits = total = 0
    for t in range(8):                            # every valid anchor with a future at t+k
        if t + k >= 8:
            continue
        W = fit_head(t, k)
        pred = unit(Z[te, t, :] @ W)              # predicted future latent
        Zt = unit(Z[te, t + k, :])                # true latents at row t+k
        M = len(te)
        for _ in range(n_trials_per):
            i = rng.integers(M)
            negs = Zt[rng.integers(M, size=n_neg)]
            cand = np.vstack([Zt[i][None, :], negs])   # 1 positive + n_neg negatives
            if np.argmax(cand @ pred[i]) == 0:    # cosine InfoNCE picks index 0?
                hits += 1
            total += 1
    return hits / total

print("accuracy vs prediction gap k (N=16):")
for k in [1, 2, 3, 4, 5]:
    print(" k", k, round(accuracy(k, 16), 3))

print("accuracy vs number of negatives (k=1):")
for n in [1, 4, 8, 16, 32, 64]:
    print(" N", n, "chance", round(1 / (n + 1), 3), "acc", round(accuracy(1, n), 3))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A CPC step has a true future score $z_{t+k}^\top W_k c_t = 2.0$ and three negatives with scores $1.0, 0.0, -1.0$. Find the InfoNCE loss for this position (use natural log).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Exponentiate every score, including the true one: $e^{2.0}, e^{1.0}, e^{0.0}, e^{-1.0}$. — _The denominator of InfoNCE sums $\exp(\text{score})$ over the whole candidate set, true plus negatives._
- Add them up to get the normalizer $Z$. — _This turns the scores into a softmax — a probability distribution over candidates._
- Compute the true future's probability $e^{2.0}/Z$, then take $-\log$ of it. — _InfoNCE is the cross-entropy of picking the one correct candidate._

**Answer:** $e^{2.0} \approx 7.39$, $e^{1.0} \approx 2.72$, $e^{0.0} = 1.0$, $e^{-1.0} \approx 0.37$. Sum $Z \approx 11.48$. True probability $= 7.39/11.48 \approx 0.644$. Loss $= -\log(0.644) \approx 0.440$. The model is fairly confident but the hard negative at score 1.0 still costs it some loss.

</details>

**Problem 2.** Why does adding more negatives generally improve the features CPC learns, and what does the bound $I(z_{t+k};c_t) \ge \log|N| - \mathcal{L}_{\text{InfoNCE}}$ have to do with it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Look at the $\log|N|$ term in the lower bound on mutual information. — _$|N|$ is the size of the candidate set; more negatives means a larger $|N|$._
- Note that a bigger $|N|$ raises the ceiling the bound can reach. — _The tightest the bound can ever certify is about $\log|N|$ bits of mutual information._

**Answer:** InfoNCE is a lower bound on the mutual information $I(z_{t+k};c_t)$, and that bound is capped near $\log|N|$. With few negatives the bound saturates early — even a great model cannot prove it captured much information. Adding negatives raises $\log|N|$, so the contrastive task can certify (and push the model toward) more mutual information between context and future. More negatives also means harder distractors on average, so the quiz teaches more. The cost is compute and memory.

</details>